In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf latex2sympy2 sympy faiss-cpu trafilatura openai-whisper

In [ ]:
import os, sys, re, time, torch
import sympy as sp
from sympy import symbols, simplify, N

BASE_DIR    = '/content/gdrive/MyDrive/NLP_assignment'
PACKAGE_DIR = os.path.join(BASE_DIR, 'millionaire_client')

if not os.path.exists(BASE_DIR):
    print(f"Error path {BASE_DIR} not found. Please check your Google Drive paths.")

sys.path.append(BASE_DIR)
print("Environment ready")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch, gc

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

print("Loading 7B planner Qwen in four bit mode")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
).eval()

print(f"Planner model loaded {MODEL_ID}")
if torch.cuda.is_available():
    print(f"GPU memory allocated {torch.cuda.memory_allocated() / 1e9:.2f} GB")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ---- Shared Speech Mode / Whisper Loader ----
import whisper
import traceback
from pathlib import Path

# Load order is intentional: Qwen 7B first, Whisper turbo last.
# Pass mode="text" or mode="speech" directly in each game run cell.
WHISPER_MODEL_SIZE = globals().get("WHISPER_MODEL_SIZE", "turbo")
# The news notebook keeps the 7B planner in memory, so the fallback list steps down if CUDA is tight.
WHISPER_FALLBACK_MODEL_SIZES = globals().get("WHISPER_FALLBACK_MODEL_SIZES", ["medium", "small", "base"])
WHISPER_DEVICE = globals().get("WHISPER_DEVICE", "cuda")
WHISPER_LANGUAGE = globals().get("WHISPER_LANGUAGE", "en")
WHISPER_FP16 = globals().get("WHISPER_FP16", True)
WHISPER_RETRY_EMPTY_OPTIONS = globals().get("WHISPER_RETRY_EMPTY_OPTIONS", True)
WHISPER_MIN_TRANSCRIPT_CHARS = globals().get("WHISPER_MIN_TRANSCRIPT_CHARS", 2)
SAVE_SPEECH_AUDIO = globals().get("SAVE_SPEECH_AUDIO", True)
DISPLAY_SPEECH_AUDIO = globals().get("DISPLAY_SPEECH_AUDIO", False)
SPEECH_AUDIO_DIR = globals().get("SPEECH_AUDIO_DIR", "/content/gdrive/MyDrive/NLP_assignment/news_speech_game_audio")
LOAD_WHISPER_IN_SHARED_MODEL_CELL = globals().get("LOAD_WHISPER_IN_SHARED_MODEL_CELL", True)

_WHISPER_MODEL_CACHE = globals().setdefault("_WHISPER_MODEL_CACHE", {})
ACTIVE_WHISPER_MODEL_SIZE = globals().get("ACTIVE_WHISPER_MODEL_SIZE", None)
ACTIVE_WHISPER_DEVICE = globals().get("ACTIVE_WHISPER_DEVICE", None)


def print_cuda_memory(label):
    try:
        if not torch.cuda.is_available():
            print(f"{label}: CUDA not available")
            return
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        gb = 1024 ** 3
        print(
            f"{label}: "
            f"free={free_bytes / gb:.2f}GB | "
            f"total={total_bytes / gb:.2f}GB | "
            f"allocated={torch.cuda.memory_allocated() / gb:.2f}GB | "
            f"reserved={torch.cuda.memory_reserved() / gb:.2f}GB"
        )
    except Exception as exc:
        print(f"{label}: CUDA memory check unavailable ({exc})")


def load_whisper_model_for_speech():
    """Load Whisper once for speech mode and return the cached model afterwards."""

    requested_device = WHISPER_DEVICE
    device_name = requested_device if requested_device == "cpu" or torch.cuda.is_available() else "cpu"
    candidate_sizes = [WHISPER_MODEL_SIZE]
    if device_name == "cuda":
        for fallback_size in WHISPER_FALLBACK_MODEL_SIZES:
            if fallback_size not in candidate_sizes:
                candidate_sizes.append(fallback_size)

    last_oom = None
    for model_size in candidate_sizes:
        cache_key = (model_size, device_name)
        if cache_key in _WHISPER_MODEL_CACHE:
            globals()["ACTIVE_WHISPER_MODEL_SIZE"] = model_size
            globals()["ACTIVE_WHISPER_DEVICE"] = device_name
            return _WHISPER_MODEL_CACHE[cache_key], device_name

        try:
            if device_name == "cuda":
                torch.cuda.empty_cache()
            print_cuda_memory(f"Before Whisper {model_size} load")
            print(f"Loading Whisper {model_size!r} on {device_name}...")
            started_at = time.time()
            _WHISPER_MODEL_CACHE[cache_key] = whisper.load_model(model_size, device=device_name)
            globals()["ACTIVE_WHISPER_MODEL_SIZE"] = model_size
            globals()["ACTIVE_WHISPER_DEVICE"] = device_name
            print(f"Whisper ready in {time.time() - started_at:.1f}s")
            print_cuda_memory(f"After Whisper {model_size} load")
            return _WHISPER_MODEL_CACHE[cache_key], device_name
        except RuntimeError as exc:
            message = str(exc).lower()
            if device_name == "cuda" and ("out of memory" in message or "cuda" in message):
                last_oom = RuntimeError(str(exc))
                traceback.clear_frames(exc.__traceback__)
                print(f"Whisper {model_size!r} did not fit on CUDA; trying fallback.")
                _WHISPER_MODEL_CACHE.pop(cache_key, None)
                del exc
                gc.collect()
                torch.cuda.empty_cache()
                try:
                    torch.cuda.ipc_collect()
                except Exception:
                    pass
                print_cuda_memory(f"After Whisper {model_size} OOM cleanup")
                continue
            raise

    if device_name == "cuda":
        print("Whisper did not fit on CUDA; retrying requested model on CPU.")
        device_name = "cpu"
        cache_key = (WHISPER_MODEL_SIZE, device_name)
        if cache_key not in _WHISPER_MODEL_CACHE:
            _WHISPER_MODEL_CACHE[cache_key] = whisper.load_model(WHISPER_MODEL_SIZE, device=device_name)
        globals()["ACTIVE_WHISPER_MODEL_SIZE"] = WHISPER_MODEL_SIZE
        globals()["ACTIVE_WHISPER_DEVICE"] = device_name
        return _WHISPER_MODEL_CACHE[cache_key], device_name

    raise RuntimeError("Could not load Whisper model") from last_oom


def clean_whisper_text(text):
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    return text.strip(' \"')


def strip_speech_noise(text):
    text = clean_whisper_text(text)
    if not text:
        return ""

    hallucination_phrases = [
        r"\bthanks? for watching[.!?]*",
        r"\bthank you for watching[.!?]*",
        r"\bbut you all too much for me to download[.!?]*",
        r"\byou all too much for me to download[.!?]*",
    ]
    for pattern in hallucination_phrases:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)

    laughter_or_filler = (
        r"(?:\b(?:a?ha(?:ha)+|ha|he(?:he)+h?|ehe(?:he)+h?|ah+|eh+|uh+|um+|ahem|pfft+)\b"
        r"[\s,.;:!?-]*)+"
    )
    previous = None
    while previous != text:
        previous = text
        text = re.sub(laughter_or_filler, " ", text, flags=re.IGNORECASE)

    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"(?:^|\s)[,.;:!?-]+(?=\s|$)", " ", text)
    text = re.sub(r"\s+", " ", text).strip(" ,.;:!?-")
    return text


def clean_question_transcript(text):
    text = strip_speech_noise(text)
    text = re.sub(r"^(?:oh|uh|um|ahem)[,!.?\s]+", "", text, flags=re.IGNORECASE).strip()
    return clean_whisper_text(text)


def clean_option_transcript(text, letter):
    text = strip_speech_noise(text)
    patterns = [
        rf"^Option\s*{letter}\s*[\.:,\)]?\s*",
        r"^Option\s*[A-D]\s*[\.:,\)]?\s*",
        rf"^{letter}\s*[\.:\)]\s*",
    ]
    for pattern in patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE).strip()
    return strip_speech_noise(text)


def transcript_has_content(text):
    return len(re.sub(r"[^A-Za-z0-9]", "", text or "")) >= WHISPER_MIN_TRANSCRIPT_CHARS


def maybe_display_audio(audio_bytes):
    if not DISPLAY_SPEECH_AUDIO:
        return
    try:
        from IPython.display import Audio, display
        display(Audio(audio_bytes))
    except Exception as exc:
        print(f"Could not display audio inline: {exc}")


def save_speech_audio(audio_bytes, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as handle:
        handle.write(audio_bytes)
    return path


def transcribe_audio_file(audio_path, initial_prompt=None, is_option=False, letter=None):
    whisper_model, whisper_device = load_whisper_model_for_speech()
    fp16 = bool(WHISPER_FP16 and whisper_device == "cuda")
    attempts = [
        {
            "initial_prompt": initial_prompt,
            "temperature": 0.0,
            "no_speech_threshold": 0.95,
            "logprob_threshold": -1.5,
            "compression_ratio_threshold": 2.8,
        },
        {
            "initial_prompt": None,
            "temperature": 0.0,
            "no_speech_threshold": 1.0,
            "logprob_threshold": -2.0,
            "compression_ratio_threshold": 3.5,
            "suppress_blank": False,
        },
        {
            "initial_prompt": None,
            "temperature": 0.2,
            "no_speech_threshold": 1.0,
            "logprob_threshold": -2.0,
            "compression_ratio_threshold": 3.5,
            "suppress_blank": False,
        },
    ]
    if not (is_option and WHISPER_RETRY_EMPTY_OPTIONS):
        attempts = attempts[:1]

    best_text = ""
    best_raw_text = ""
    for attempt_index, attempt in enumerate(attempts, start=1):
        kwargs = {
            "language": WHISPER_LANGUAGE,
            "task": "transcribe",
            "fp16": fp16,
            "condition_on_previous_text": False,
            **attempt,
        }
        if float(kwargs.get("temperature", 0.0)) == 0.0:
            kwargs["beam_size"] = 5
        else:
            kwargs["best_of"] = 5
        prompt = kwargs.pop("initial_prompt", None)
        if prompt:
            kwargs["initial_prompt"] = prompt

        result = whisper_model.transcribe(str(audio_path), **kwargs)
        raw_text = clean_whisper_text(result.get("text", ""))
        cleaned_text = clean_option_transcript(raw_text, letter or "") if is_option else clean_question_transcript(raw_text)

        if raw_text and (not best_raw_text or len(raw_text) > len(best_raw_text)):
            best_raw_text = raw_text
        if cleaned_text and (not best_text or len(cleaned_text) > len(best_text)):
            best_text = cleaned_text

        if transcript_has_content(cleaned_text):
            if raw_text != cleaned_text:
                print(f"Cleaned transcript noise: {raw_text!r} -> {cleaned_text!r}")
            return cleaned_text

        if is_option and attempt_index < len(attempts):
            print(f"Empty/low-content option transcript from {Path(audio_path).name}; retrying Whisper pass {attempt_index + 1}...")

    if best_raw_text and best_raw_text != best_text:
        print(f"Cleaned transcript noise: {best_raw_text!r} -> {best_text!r}")
    return best_text


def transcribe_speech_question(game):
    question = game.current_question
    if question is None:
        return None, {"error": "No active question returned by server."}

    audio_dir = Path(SPEECH_AUDIO_DIR)
    level = game.current_level
    session_id = game.session_id
    transcript = {
        "mode": "speech",
        "session_id": session_id,
        "level": level,
        "audio_files": {},
        "question": None,
        "options": [],
        "whisper_model_size": globals().get("ACTIVE_WHISPER_MODEL_SIZE") or WHISPER_MODEL_SIZE,
        "whisper_device": globals().get("ACTIVE_WHISPER_DEVICE") or WHISPER_DEVICE,
    }
    started_at = time.time()

    def option_letter(index):
        letters = globals().get("LETTERS", "ABCD")
        return letters[index] if index < len(letters) else chr(65 + index)

    def audio_path_for(kind, letter=None):
        if kind == "question":
            filename = f"session_{session_id}_level_{level}_question.wav"
        else:
            filename = f"session_{session_id}_level_{level}_option_{letter}.wav"
        if SAVE_SPEECH_AUDIO:
            return audio_dir / filename
        return Path("/tmp") / filename

    print("Fetching question audio...")
    question_audio = game.fetch_audio_question()
    question_path = audio_path_for("question")
    save_speech_audio(question_audio, question_path)
    transcript["audio_files"]["question"] = str(question_path)
    maybe_display_audio(question_audio)

    option_paths = []
    option_count = len(getattr(question, "options", []) or []) or 4
    for index in range(option_count):
        letter = option_letter(index)
        print(f"Fetching option {letter} audio...")
        option_audio = game.fetch_audio_option_next()
        option_path = audio_path_for("option", letter)
        save_speech_audio(option_audio, option_path)
        transcript["audio_files"][letter] = str(option_path)
        option_paths.append((letter, option_path))
        maybe_display_audio(option_audio)

    try:
        game.refresh_state()
        refreshed_question = game.current_question
        if refreshed_question is not None:
            question = refreshed_question
    except Exception as exc:
        print(f"Could not refresh game state after speech audio delivery: {exc}")

    print("Transcribing question audio...")
    question_text = transcribe_audio_file(question_path, initial_prompt="A multiple choice trivia question.")
    question.text = question_text
    transcript["question"] = question_text
    print("Question transcript:", question_text)

    option_texts = []
    for index, (letter, option_path) in enumerate(option_paths):
        option_text = transcribe_audio_file(option_path, initial_prompt=None, is_option=True, letter=letter)
        option_texts.append(option_text)
        transcript["options"].append({"letter": letter, "audio_file": str(option_path), "text": option_text})
        print(f"Option {letter} transcript: {option_text}")

    for index, option in enumerate(question.options):
        if index < len(option_texts):
            option.text = option_texts[index]

    transcript["transcription_seconds"] = time.time() - started_at
    try:
        transcript["seconds_left_after_audio"] = seconds_available(game)
    except NameError:
        transcript["seconds_left_after_audio"] = game.time_remaining
    transcript["whisper_model_size"] = globals().get("ACTIVE_WHISPER_MODEL_SIZE") or WHISPER_MODEL_SIZE
    transcript["whisper_device"] = globals().get("ACTIVE_WHISPER_DEVICE") or WHISPER_DEVICE
    return question, transcript


if LOAD_WHISPER_IN_SHARED_MODEL_CELL:
    print("Loading Whisper last, after Qwen 7B is already loaded...")
    speech_whisper_model, speech_whisper_device = load_whisper_model_for_speech()
    print(f"Shared speech model ready: Whisper {globals().get('ACTIVE_WHISPER_MODEL_SIZE')} on {speech_whisper_device}")
else:
    print("Shared speech model preload skipped. Speech loops will load Whisper before starting the timer.")



In [ ]:
import os
import re
import json
import math
import faiss
import numpy as np
import requests
import warnings
import urllib.parse
import concurrent.futures
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
from bs4 import BeautifulSoup
import trafilatura
from sentence_transformers import SentenceTransformer

# Minimize noisy warnings in notebook execution
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.simplefilter(action="ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore")

# SERPER API key: read from environment, fallback kept for compatibility
SERPER_API_KEY = os.getenv("SERPER_API_KEY", "")


# Web scraping helpers and article extraction
def extract_article_text(link):
    """Extract article body from URL with trafilatura + HTML fallback."""
    real_url = link
    if "bing.com" in link and "url=" in link.lower():
        parsed = urllib.parse.urlparse(link)
        qs = urllib.parse.parse_qs(parsed.query)
        if "url" in qs:
            real_url = qs["url"][0]

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Referer": "https://www.google.com/",
        "DNT": "1",
        "Upgrade-Insecure-Requests": "1",
    }

    try:
        downloaded = trafilatura.fetch_url(real_url)
        if downloaded:
            extracted = trafilatura.extract(downloaded)
            if extracted and len(extracted) > 200:
                return extracted[:8000]
    except Exception:
        pass

    try:
        resp = requests.get(real_url, headers=headers, timeout=4, allow_redirects=True)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "html.parser")
            for elemento in soup(["script", "style", "nav", "header", "footer", "aside"]):
                elemento.extract()
            paragraphs = soup.find_all(["p", "li"])
            text = " ".join([p.get_text(strip=True) for p in paragraphs if len(p.get_text(strip=True)) > 30])
            if text:
                return text[:8000]
    except Exception:
        pass

    return ""


# Primary search wrapper using the Serper News API
def extract_date_range_from_question(question_text):
    match = re.search(r"\b(202\d)-(\d{2})-(\d{2})\b", question_text)
    if match:
        year, month, day = match.groups()
        try:
            date_obj = datetime.strptime(f"{year}-{month}-{day}", "%Y-%m-%d")
            date_from = (date_obj - timedelta(days=1)).strftime("%m/%d/%Y")
            date_to = (date_obj + timedelta(days=1)).strftime("%m/%d/%Y")
            return f"cdr:1,cd_min:{date_from},cd_max:{date_to}"
        except Exception:
            pass
    return ""


def serper_news_search(query, tbs_date_filter=""):
    print(f"\n[SEARCH] Serper news query: '{query}'")
    if tbs_date_filter:
        print(f"        Applied date range: {tbs_date_filter}")

    if not SERPER_API_KEY:
        print("[SEARCH] SERPER_API_KEY not found — skipping Serper primary search.")
        return ""

    url = "https://google.serper.dev/news"
    payload = {"q": query, "num": 6}
    if tbs_date_filter:
        payload["tbs"] = tbs_date_filter

    headers = {"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"}

    try:
        response = requests.post(url, headers=headers, data=json.dumps(payload), timeout=10)
        if response.status_code == 200:
            results = response.json().get("news", [])

            if not results and tbs_date_filter:
                print("        [SEARCH] No results for constrained date; retrying without date.")
                payload.pop("tbs", None)
                resp_no_date = requests.post(url, headers=headers, data=json.dumps(payload), timeout=10)
                results = resp_no_date.json().get("news", [])

            if not results:
                print("        [SEARCH] No news entries returned; attempting organic search fallback.")
                url_search = "https://google.serper.dev/search"
                resp_search = requests.post(url_search, headers=headers, data=json.dumps(payload), timeout=10)
                results = resp_search.json().get("organic", [])

            top_links = [r.get("link") for r in results[:2] if r.get("link")]
            extracted_texts = {}

            def scrape_worker(link):
                return extract_article_text(link)

            with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
                future_to_link = {executor.submit(scrape_worker, link): link for link in top_links}
                for future in concurrent.futures.as_completed(future_to_link):
                    link = future_to_link[future]
                    try:
                        extracted_texts[link] = future.result()
                    except Exception:
                        extracted_texts[link] = ""

            context = ""
            for r in results[:4]:
                title = r.get("title", "")
                snippet = r.get("snippet", "")
                date_str = r.get("date", "")
                link = r.get("link", "")

                context += f"Title: {title}\nDate: {date_str}\nSummary: {snippet}\n"
                if link in extracted_texts and extracted_texts[link]:
                    context += f"EXTENDED FULL TEXT: {extracted_texts[link][:2500]}...\n"
                context += "\n"

            return context
    except Exception as e:
        print(f"[SEARCH] Serper error: {e}")

    return ""


# Secondary retrieval: Bing RSS + FAISS
class NewsRealTimeRAG:
    def __init__(self):
        print("[RAG] Starting Bing RSS + FAISS fallback engine...")
        self.embedder = SentenceTransformer("all-MiniLM-L6-v2")
        self.cache = {}

    def chunk_text(self, text, chunk_size=600, overlap=150):
        text = re.sub(r"\s+", " ", text)
        sentences = re.split(r"(?<=[.])\s+", text)
        chunks = []
        current_chunk = ""
        for sentence in sentences:
            if len(current_chunk) + len(sentence) <= chunk_size:
                current_chunk += " " + sentence
            else:
                if current_chunk.strip():
                    chunks.append(current_chunk.strip())
                current_chunk = sentence
        if current_chunk.strip():
            chunks.append(current_chunk.strip())
        return chunks

    def retrieve(self, news_query, faiss_query, top_k=6):
        query_variants = [news_query]
        words = news_query.split()
        if len(words) > 3:
            query_variants.append(" ".join(words[:-1]))
        if len(words) > 2:
            query_variants.append(" ".join(words[:2]))

        all_chunks = []
        seen_urls = set()
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

        for q in query_variants:
            if not q.strip():
                continue
            if q in self.cache:
                all_chunks.extend(self.cache[q])
                break

            query_chunks = []
            try:
                encoded_query = urllib.parse.quote(q)
                url = f"https://www.bing.com/news/search?q={encoded_query}&format=rss"
                response = requests.get(url, headers=headers, timeout=10)

                if response.status_code == 200:
                    root = ET.fromstring(response.text)
                    items = root.findall(".//channel/item")

                    valid_items = []
                    for item in items:
                        link = item.find("link").text if item.find("link") is not None else ""
                        if link and link not in seen_urls:
                            seen_urls.add(link)
                            valid_items.append((item, link))
                        if len(valid_items) >= 3:
                            break

                    def worker(data):
                        it, lnk = data
                        title = it.find("title").text if it.find("title") is not None else ""
                        desc = it.find("description").text if it.find("description") is not None else ""
                        desc_clean = re.sub("<[^<]+>", " ", desc)
                        article_body = extract_article_text(lnk)
                        full_text = f"{title}. {desc_clean}. {article_body}"
                        return self.chunk_text(full_text)

                    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
                        for chunks in executor.map(worker, valid_items):
                            if chunks:
                                query_chunks.extend(chunks)

                    self.cache[q] = query_chunks
                    all_chunks.extend(query_chunks)
                    if all_chunks:
                        break
            except Exception:
                continue

        if not all_chunks:
            return ""

        unique_chunks = []
        seen_chunks = set()
        for chunk in all_chunks:
            fingerprint = chunk[:100].strip()
            if fingerprint not in seen_chunks:
                seen_chunks.add(fingerprint)
                unique_chunks.append(chunk)

        if not unique_chunks:
            return ""

        embeddings = self.embedder.encode(unique_chunks, convert_to_numpy=True)
        index = faiss.IndexFlatL2(embeddings.shape[1])
        index.add(embeddings)

        query_vector = self.embedder.encode([faiss_query], convert_to_numpy=True)
        _, indices = index.search(query_vector, min(top_k, len(unique_chunks)))

        docs = [unique_chunks[idx] for idx in indices[0][:6]]
        return "\n\n".join(docs)


rag_backup_engine = NewsRealTimeRAG()


# Create compact search queries for news retrieval
def generate_news_query_with_options(question_text, options):
    query_prompt = f"""[INST] You are an elite OSINT Intelligence Search Specialist.
Your job is to create a highly effective Google search query (MAX 5 WORDS) to find the specific news article.

CRITICAL RULES:
1. EXTRACT THE CORE EVENT: Focus ONLY on the unique proper nouns and subjects from the QUESTION.
2. STRICT OPTION BAN: NEVER include words from the options in your query.
3. ONLY NOUNS: Output only raw keywords separated by spaces. No verbs.

Now, generate the keywords for this question:
Question: {question_text}
Keywords: [/INST]"""

    inputs = tokenizer(query_prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=25,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    keywords = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return keywords


# Answer selection: LLM decision logic using dual retrieval engines
def extract_letter(text):
    if "FINAL ANSWER: NONE" in text.upper():
        return "N"
    match = re.search(r"FINAL ANSWER:\s*([ABCD])", text, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    matches = re.findall(r"\b([ABCD])\b", text.upper())
    if matches:
        return matches[-1]
    return "A"


def interrogate_llm(context, question, instruction_set):
    prompt_speech = f"""[INST] You are an elite News Analyst taking a multiple-choice test on current events.

Context:
{context}

Question:
{question.text}
A) {question.options[0].text}
B) {question.options[1].text}
C) {question.options[2].text}
D) {question.options[3].text}

Instructions:
1. {instruction_set}
2. CONNECT THE DOTS: You MUST combine information from multiple snippets to find the correct option.
3. CRITICAL RULE: If the provided context does NOT contain the answer, you must output 'FINAL ANSWER: NONE'. Do NOT guess.
4. ABSOLUTE FORMAT RULE: You MUST output ONLY the single letter (A, B, C, or D) after 'FINAL ANSWER:'. NEVER write the full text of the option.

You MUST use EXACTLY this format:
Eval: [In max 15 words, justify the option]
FINAL ANSWER: [A, B, C, D, or NONE]
[/INST]Eval: """

    inputs = tokenizer(prompt_speech, return_tensors="pt", truncation=True, max_length=3072).to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)


def choose_answer(question):
    if len(question.options) < 4:
        return question.options[0].id, "A", "fallback"

    news_query = generate_news_query_with_options(question.text, question.options)
    tbs_filter = extract_date_range_from_question(question.text)

    is_negative_question = "NOT " in question.text.upper() or "EXCEPT" in question.text.upper() or "FALSE" in question.text.upper()
    if is_negative_question:
        instruction_set = "This is a NEGATIVE question. Find the ONE option that is completely missing or denied by the text."
    else:
        instruction_set = "Read carefully. MAXIMUM SPECIFICITY RULE: If multiple options appear as nested locations, choose the most specific entity. STRICT ANTI-GUESSING: If the core concepts of the options are completely missing, output 'FINAL ANSWER: NONE'."

    letter = "N"
    final_text = ""

    # First attempt: query Serper primary search
    context = serper_news_search(news_query, tbs_filter)
    if context.strip():
        print("\n" + "-" * 40)
        print("[PRIMARY CONTEXT] Retrieved from Serper:")
        print(context)
        print("-" * 40 + "\n")

        final_text = interrogate_llm(context, question, instruction_set)
        letter = extract_letter(final_text)
        print(f"[PRIMARY OUTPUT]\n{final_text}")

    # Fallback: use Bing RSS + FAISS retrieval if the primary search fails
    if letter == "N":
        print("\n[RAG] Primary returned NONE; engaging backup retriever...")
        faiss_query = f"{question.text} {' '.join([opt.text for opt in question.options])}"
        bing_context = rag_backup_engine.retrieve(news_query=news_query, faiss_query=faiss_query, top_k=6)

        if bing_context.strip():
            print("\n" + "-" * 40)
            print("[BACKUP CONTEXT] Retrieved from Bing RSS:")
            print(bing_context)
            print("-" * 40 + "\n")

            final_text = interrogate_llm(bing_context, question, instruction_set)
            letter = extract_letter(final_text)
            print(f"[BACKUP OUTPUT]\n{final_text}")
        else:
            print("\n[RAG] Backup retriever returned no matches.")

    if letter == "N":
        print("\n[RAG] No answer found; defaulting to option A for safety.")
        letter = "A"

    try:
        idx = ["A", "B", "C", "D"].index(letter)
    except ValueError:
        idx = 0
        letter = "A"

    return question.options[idx].id, letter, final_text

In [ ]:
from millionaire_client import MillionaireClient
from millionaire_client.exceptions import TimeoutError, RateLimitError


def play_game(competition_id=5, mode="text"):
    if mode not in {"text", "speech"}:
        raise ValueError('mode must be either "text" or "speech"')
    if mode == "speech":
        load_whisper_model_for_speech()

    API_URL = (__import__("os").getenv("POLI_MILLIONAIRE_API_URL") or input("PoliMillionaire API URL: ").strip())
    client = MillionaireClient(API_URL)
    user = client.login((__import__("os").getenv("POLI_MILLIONAIRE_USERNAME") or input("PoliMillionaire username: ").strip()), (__import__("os").getenv("POLI_MILLIONAIRE_PASSWORD") or __import__("getpass").getpass("PoliMillionaire password: ").strip()))
    print(f"Logged in as {user.username}")

    game = client.game.start(competition_id=competition_id, mode=mode)

    while game.in_progress:
        speech_transcription = None
        if game.mode == "speech":
            question, speech_transcription = transcribe_speech_question(game)
        else:
            question = game.current_question
        if question is None:
            break

        print(f"Level: {game.current_level} | Question: {question.text}")
        if speech_transcription:
            seconds_left = speech_transcription.get("seconds_left_after_audio")
            print(
                "Speech transcription:",
                f"{speech_transcription.get('transcription_seconds', 0.0):.1f}s",
                f"| {seconds_left:.1f}s left after audio" if seconds_left is not None else "| time left n/a",
                f"| Whisper {speech_transcription.get('whisper_model_size')} on {speech_transcription.get('whisper_device')}",
            )
        for i, opt in enumerate(question.options):
            print(f"Option {chr(65+i)} {opt.text}")

        t0 = time.time()
        option_id, letter, method = choose_answer(question)
        t1 = time.time()

        print(f"Predicted {letter} Method {method} Time taken {t1-t0:.2f} seconds")

        try:
            result = game.answer(option_id)
            print(f"Correct {result.correct} Earned {result.earned_amount}")
        except TimeoutError:
            print("Timed out generation took more than thirty seconds")
            break
        except RateLimitError:
            print("Rate limited waiting five seconds")
            time.sleep(5)
            result = game.answer(option_id)
            print(f"Correct {result.correct} Earned {result.earned_amount}")

        if result.game_over:
            break
        time.sleep(1)

    print(f"Game over final score {game.earned_amount}")
    return game


In [ ]:
game = play_game(mode="speech")


In [ ]:
earned = []
for i in range (30):
    game = play_game(mode="text")
    earned.append(game.earned_amount)
max_earned = max(earned)
print("Max earned:", max_earned)